# 3.1.4 — Adversarial Rollout Validation

Checks that `opp_rollout_mode` changes rankings as expected. Edit `SEASON`, `TEST_MAP`, and `MY_PICKS`/`OPP_PICKS` as needed.

**Pass criterion:** At least one brawler shifts rank between `weighted` and `greedy` mode. If rankings are identical across all modes, the rollout is not flowing through — likely a FM discrimination issue.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "../src")

import numpy as np
from recommend import recommend
from fm_integration import FMEvaluator
from fm_model import FMInference
from matchup_db import MatchupDB

SEASON = "s48"          # "s42" | "s48"
DATA_DIR = Path("..").resolve() / "data" / SEASON

evaluator = FMEvaluator(FMInference.load(DATA_DIR / "fm_model.pkl"))
db        = MatchupDB.load(DATA_DIR / "matchup_db.pkl")
schema    = evaluator._fm.schema

# ── Test state (edit these) ───────────────────────────────────────────────────
TEST_MAP  = schema.maps[22]
TEST_MODE = schema.modes[0]
MY_PICKS  = []
OPP_PICKS = ["Nani"]
IS_FIRST  = False
SKILL_NS  = 4

N_SIMS            = 50_000
N_TOP             = 10
MIN_PICK_RATE     = 0.00   # brawlers below this map/mode/tier pick rate excluded from tree
MIN_COUNTER_GAMES = 50     # matchup entries with fewer games treated as 0.5 neutral

print(f"{SEASON} | {TEST_MAP} ({TEST_MODE}) | P{'1' if IS_FIRST else '2'} | {N_SIMS} sims")
print(f"min_pick_rate={MIN_PICK_RATE}  min_counter_games={MIN_COUNTER_GAMES}")

Run `recommend()` under three modes with a fixed seed, then print rankings side by side.
Brawlers with a hard counter should rank lower as the mode becomes more adversarial.

In [ ]:
MODES = [
    ("weighted",   dict(opp_rollout_mode="weighted", opp_counter_weight=0.8)),
    ("top_k k=6",  dict(opp_rollout_mode="top_k", opp_top_k=6)),
    ("top_k k=3",  dict(opp_rollout_mode="top_k", opp_top_k=3)),
    ("greedy k=1", dict(opp_rollout_mode="top_k", opp_top_k=1)),
]

results = {}
for label, kwargs in MODES:
    r = recommend(
        my_picks=MY_PICKS, opp_picks=OPP_PICKS, mode=TEST_MODE, map_name=TEST_MAP,
        skill_ns=SKILL_NS, is_first_pick=IS_FIRST, n_simulations=N_SIMS, n_top=N_TOP,
        min_pick_rate=MIN_PICK_RATE, min_counter_games=MIN_COUNTER_GAMES,
        evaluator=evaluator, db=db, rng=np.random.default_rng(42), **kwargs,
    )
    results[label] = r

# ── Side-by-side table ────────────────────────────────────────────────────────
labels = [l for l, _ in MODES]
COL = 28
print("═" * (COL * 3))
print("".join(f"{l:^{COL}}" for l in labels))
print("".join(f"  Q={results[l].layer2['root_q']:.4f}{'':<{COL-10}}" for l in labels))
print("─" * (COL * 3))
for rank in range(1, N_TOP + 1):
    row = ""
    for l in labels:
        picks = results[l].top_picks
        if rank <= len(picks):
            p = picks[rank - 1]
            row += f"  {rank}. {p['brawler']:<14}{p['estimated_win_prob']:.3f}{'':<{COL-23}}"
        else:
            row += " " * COL
    print(row)
print("═" * (COL * 3))

# ── Rank shifts (weighted → greedy) ──────────────────────────────────────────
w = {p['brawler']: i for i, p in enumerate(results[labels[0]].top_picks)}
g = {p['brawler']: i for i, p in enumerate(results[labels[-1]].top_picks)}
shifts = sorted(
    [(b, w.get(b, N_TOP)+1, g.get(b, N_TOP)+1) for b in set(w)|set(g) if w.get(b,N_TOP) != g.get(b,N_TOP)],
    key=lambda x: abs(x[2]-x[1]), reverse=True,
)
print("\nRank shifts (weighted → greedy):")
for b, wr, gr in shifts:
    print(f"  {b:<16} #{wr} → #{gr}  {'↓ worse' if gr > wr else '↑ better'} under greedy")
if not shifts:
    print("  None — modes produce identical rankings (investigate FM discrimination)")

For the brawler with the biggest rank drop under greedy mode, check who counters it in the matchup DB.
The drop should be explained by a real high-win-rate counter (with meaningful game count).

In [ ]:
from matchup_db import skill_ns_to_tier

target = shifts[0][0] if shifts else results[labels[0]].top_picks[0]["brawler"]
tier   = skill_ns_to_tier(SKILL_NS, db.skill_tier_boundaries) if db.skill_tier_boundaries else 1

rows = [
    (opp, e["win_rate"], e["n"])
    for opp in schema.vocab
    if opp != target
    for e in [db.counter_lookup(opp, target, TEST_MODE, TEST_MAP, tier)]
    if e is not None and e["n"] >= MIN_COUNTER_GAMES
]
rows.sort(key=lambda x: x[1], reverse=True)

print(f"Counters of {target} on {TEST_MAP} (tier={tier}, min_games={MIN_COUNTER_GAMES}):")
for opp, wr, n in rows[:8]:
    print(f"  {opp:<16} win_rate={wr:.3f}  n={n}")
if not rows:
    print("  No matchups meet the min_games threshold — try lowering MIN_COUNTER_GAMES.")

## 3.2.5 — PUCT Prior Validation

Check that PUCT priors concentrate opponent-node exploration toward counter-picks, and that this shifts my recommendations away from easily-countered brawlers.

**Pass criterion 1:** Under PUCT, visit counts at the opp-turn root node should correlate more strongly with `avg_counter_rate` than under plain UCB1 (Pearson r\_PUCT > r\_UCB1).

**Pass criterion 2:** A brawler that ranks highly under plain UCB1 but has a known hard counter should drop in rank (or disappear from top-10) under PUCT.

In [ ]:
# ── Pass criterion 1: visit distribution at opp-turn root node ───────────────
# State: P1, I've picked MY_BRAWLER. Next pick is opp-turn → root is an opp node.
# Run 10k sims with PUCT (alpha=0.7) and without (alpha=0.0).
# Expect visit counts to correlate with counter_rate more strongly under PUCT.

from recommend import _run_mcts
from rollout import RolloutWeightCache
from draft_state import DraftState
from matchup_db import skill_ns_to_tier

MY_BRAWLER = shifts[0][0] if shifts else results[labels[0]].top_picks[0]["brawler"]
tier = skill_ns_to_tier(SKILL_NS, db.skill_tier_boundaries) if db.skill_tier_boundaries else 1

opp_root_state = DraftState(
    my_team=frozenset({MY_BRAWLER}), opp_team=frozenset(),
    mode=TEST_MODE, map_name=TEST_MAP, skill_ns=SKILL_NS, is_first_pick=True,
)
assert opp_root_state.whose_turn == "opp", "Expected opp-turn root"

_common = dict(
    evaluator=evaluator, db=db, vocab=schema.vocab,
    n_simulations=10_000,
    my_counter_weight=0.3, opp_counter_weight=0.5,
    min_pick_rate=MIN_PICK_RATE, min_counter_games=MIN_COUNTER_GAMES,
    opp_rollout_mode="weighted", opp_top_k=3,
    my_rollout_mode="weighted", my_top_k=3,
    c=0.5,
)
root_puct = _run_mcts(opp_root_state, **_common, rng=np.random.default_rng(42), puct_alpha=0.7)
root_ucb1 = _run_mcts(opp_root_state, **_common, rng=np.random.default_rng(42), puct_alpha=0.0)

# Build (brawler, visit_count, counter_rate) table for each run.
wc = RolloutWeightCache(db, schema.vocab, min_counter_games=MIN_COUNTER_GAMES)
my_idx = [wc.brawler_idx[MY_BRAWLER]]

def _child_data(root):
    rows = []
    n = len(root._child_list)
    _, cm = wc._get(TEST_MODE, TEST_MAP, tier)
    for i in range(n):
        b = root._all_actions[i]
        bi = wc.brawler_idx[b]
        cr = float(cm[bi, my_idx[0]])
        rows.append((b, float(root._child_visits[i]), cr))
    return sorted(rows, key=lambda x: -x[1])  # sort by visits desc

puct_rows = _child_data(root_puct)
ucb1_rows = _child_data(root_ucb1)

# Print top-10 most visited for each.
COL = 40
print(f"Opp-node visit distribution  (my pick = {MY_BRAWLER})")
print(f"{'PUCT (alpha=0.7)':<{COL}}  {'UCB1 (no prior)'}")
print("─" * (COL * 2 + 2))
for (pb, pv, pcr), (ub, uv, ucr) in zip(puct_rows[:10], ucb1_rows[:10]):
    print(f"  {pb:<14} visits={pv:>5.0f}  cr={pcr:.3f}    "
          f"  {ub:<14} visits={uv:>5.0f}  cr={ucr:.3f}")

# Pearson r between visit_count and counter_rate for created children.
v_p = np.array([r[1] for r in puct_rows])
c_p = np.array([r[2] for r in puct_rows])
v_u = np.array([r[1] for r in ucb1_rows])
c_u = np.array([r[2] for r in ucb1_rows])

r_puct = float(np.corrcoef(v_p, c_p)[0, 1]) if len(v_p) > 1 else 0.0
r_ucb1 = float(np.corrcoef(v_u, c_u)[0, 1]) if len(v_u) > 1 else 0.0
print(f"\nPearson r(visits, counter_rate):  PUCT={r_puct:+.3f}  UCB1={r_ucb1:+.3f}")

# Pass criterion 1
assert r_puct > r_ucb1, (
    f"FAIL — PUCT correlation ({r_puct:.3f}) should exceed UCB1 ({r_ucb1:.3f}). "
    "PUCT priors are not concentrating exploration on counters."
)
print(f"✓ Pass criterion 1: PUCT r={r_puct:.3f} > UCB1 r={r_ucb1:.3f}")

Scatter: x = `avg_counter_rate` vs MY_BRAWLER, y = visit count at the opp-turn root node. Under PUCT the trend should be upward (high-counter-rate brawlers are visited more); under UCB1 it should be flat. Also compare my top-10 picks under PUCT vs UCB1 — brawlers with hard counters should drop in rank.

In [ ]:
import matplotlib.pyplot as plt

# ── Scatter: counter_rate vs visit_count ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, rows, label, color in [
    (axes[0], puct_rows, f"PUCT α=0.7  (r={r_puct:+.3f})", "steelblue"),
    (axes[1], ucb1_rows, f"UCB1 no prior  (r={r_ucb1:+.3f})", "tomato"),
]:
    crs = [r[2] for r in rows]
    vs  = [r[1] for r in rows]
    ax.scatter(crs, vs, c=color, alpha=0.7, s=40)
    # Trend line
    m, b = np.polyfit(crs, vs, 1)
    xs = np.linspace(min(crs), max(crs), 100)
    ax.plot(xs, m * xs + b, color=color, lw=1.5, linestyle="--")
    ax.set_xlabel("avg_counter_rate vs " + MY_BRAWLER)
    ax.set_ylabel("visit count")
    ax.set_title(label)
    ax.axvline(0.5, color="grey", lw=0.8, linestyle=":")
plt.suptitle(f"Opp-node exploration: {MY_BRAWLER} as my pick  |  {TEST_MAP}", fontsize=11)
plt.tight_layout()
plt.show()

# ── Pass criterion 2: my recommendations shift under PUCT ────────────────────
# Run from empty P1 draft state — root is my-turn, PUCT affects deeper opp nodes.
r_my_puct = recommend(
    my_picks=[], opp_picks=[], mode=TEST_MODE, map_name=TEST_MAP,
    skill_ns=SKILL_NS, is_first_pick=True, n_simulations=10_000, n_top=N_TOP,
    min_pick_rate=MIN_PICK_RATE, min_counter_games=MIN_COUNTER_GAMES,
    puct_alpha=0.7, evaluator=evaluator, db=db, rng=np.random.default_rng(42),
)
r_my_ucb1 = recommend(
    my_picks=[], opp_picks=[], mode=TEST_MODE, map_name=TEST_MAP,
    skill_ns=SKILL_NS, is_first_pick=True, n_simulations=10_000, n_top=N_TOP,
    min_pick_rate=MIN_PICK_RATE, min_counter_games=MIN_COUNTER_GAMES,
    puct_alpha=0.0, evaluator=evaluator, db=db, rng=np.random.default_rng(42),
)

puct_picks = [p["brawler"] for p in r_my_puct.top_picks]
ucb1_picks = [p["brawler"] for p in r_my_ucb1.top_picks]

print(f"My top-{N_TOP} picks (empty P1 draft, {TEST_MAP})")
print(f"{'PUCT α=0.7':<28}  {'UCB1 (no prior)'}")
print("─" * 58)
for i, (pp, up) in enumerate(zip(puct_picks, ucb1_picks), 1):
    marker = " ←" if pp != up else ""
    print(f"  {i}. {pp:<22}   {up}{marker}")

# Check: MY_BRAWLER (the easily-countered pick) ranks lower under PUCT.
puct_rank = puct_picks.index(MY_BRAWLER) + 1 if MY_BRAWLER in puct_picks else N_TOP + 1
ucb1_rank = ucb1_picks.index(MY_BRAWLER) + 1 if MY_BRAWLER in ucb1_picks else N_TOP + 1
print(f"\n{MY_BRAWLER}: UCB1 rank #{ucb1_rank}  →  PUCT rank #{puct_rank}")

assert puct_rank >= ucb1_rank, (
    f"FAIL — {MY_BRAWLER} should rank equal or lower under PUCT (got #{puct_rank} vs UCB1 #{ucb1_rank}). "
    "PUCT is not penalising easily-countered picks."
)
print(f"✓ Pass criterion 2: {MY_BRAWLER} rank #{ucb1_rank} → #{puct_rank} under PUCT")